# 2 — Channel Selection (Sequential Forward Selection)

Finds which combination of the 10 input channels gives the best GRU detection
skill (TSS), using a greedy Sequential Forward Selection search rather than the
full `2^10 - 1 = 1,023` subsets.

Greedy search evaluates `10 + 9 + ... + 1 = 55` subsets instead of 1,023. The
reduction is not only computational: with 34 SEP test events, picking the best
of 1,023 noisy TSS estimates is a multiple-comparisons problem in which the
winner is likely to be noise.

**Input:** `./final_split_data_HybridNorm` (notebook 1, Section 10).

**Output:** `./results/gru_feature_selection_sfs.txt` — one row per run
(55 subsets x 5 runs = 275 rows) — and `./final_split_data_HybridNorm_BestFeatures`
holding the winning subset.

**Method:** round 1 scores each channel alone and keeps the best; each later
round adds whichever remaining channel most improves mean TSS; round 10 is the
full set. All 55 subsets are then ranked, so the overall winner may be smaller
than the greedy path's endpoint.

GRU hyperparameters are fixed throughout, so the comparison isolates the effect
of channel choice.

**Run after:** notebook 1. **Run before:** notebook 3.


## 1. Load Hybrid-Normalized Data

In [6]:
import pickle
import numpy as np
import os

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

folder = "./final_split_data_HybridNorm"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

feature_names = ['F', 'Np', 'P4', 'P5', 'P6', 'Tp', 'V', 'Vx', 'Xl', 'Xs']
feature_index = {name: i for i, name in enumerate(feature_names)}

train_counts = np.bincount(y_train.astype(int), minlength=2)
val_counts   = np.bincount(y_val.astype(int),   minlength=2)
test_counts  = np.bincount(y_test.astype(int),  minlength=2)

print(f"Train: {X_train.shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
print(f"Val  : {X_val.shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
print(f"Test : {X_test.shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
print(f"Features ({len(feature_names)}): {feature_names}")

Train: (12455, 288, 10) | Non-SEP=12337 | SEP=118
Val  : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Features (10): ['F', 'Np', 'P4', 'P5', 'P6', 'Tp', 'V', 'Vx', 'Xl', 'Xs']


## 2. GRU Helper Functions

In [2]:
import torch
import torch.nn as nn
import time
from sklearn.metrics import confusion_matrix, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    # Probability of Detection (PoD), MEMPSEP Section 9.2 item 1.
    # Identical formula to recall -- kept as a separate named field so
    # printed/saved output matches the paper's terminology.
    pod       = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    recall    = pod
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    # HSS1 (Barnes & Leka 2008): (TP + TN - N) / P  ==  (TP - FP) / (TP + FN)
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    far = FP / (TP + FP) if (TP + FP) > 0 else float('nan')
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'far': far, 'bias': bias,
        'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'pod': pod, 'recall': recall, 'precision': precision, 'f1': f1, 'accuracy': accuracy
    }


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['brier']      = brier_score_loss(y_test, y_prob)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model

Using device: mps


## 3. Sequential Forward Selection

Each candidate subset is trained and evaluated 5 times. Every individual run is
appended to the results file as it completes, with `round`, `features`, and
`run` columns so subsets can be told apart in one file.


In [3]:
import copy

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(RESULTS_DIR, "gru_feature_selection_sfs.txt")

N_RUNS = 5

# ---------------------------------------------------------
# Fixed GRU hyperparameters (same as notebooks 1 and 3) -- held constant
# across all combinations so the comparison isolates feature choice.
# ---------------------------------------------------------
best_params = {
    "units": 64,
    "layers": 1,
    "dropout": 0.1,
    "lr": 0.001,
    "batch_size": 64
}

class_ratio = train_counts[0] / train_counts[1]

HEADER = ("round,features,run,TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,"
          "precision,f1,accuracy,brier,train_time,infer_time,far,bias")

with open(RESULTS_FILE, "w") as f:
    f.write(HEADER + "\n")


def append_result(round_idx, feat_subset, run_idx, m):
    feat_str = "|".join(feat_subset)
    line = (
        f"{round_idx},{feat_str},{run_idx},"
        f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
        f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
        f"{m['pod']:.6f},{m['recall']:.6f},{m['precision']:.6f},{m['f1']:.6f},"
        f"{m['accuracy']:.6f},{m['brier']:.6f},"
        f"{m['train_time']:.4f},{m['infer_time']:.4f},"
        f"{m['far']:.6f},{m['bias']:.6f}"
    )
    with open(RESULTS_FILE, "a") as f:
        f.write(line + "\n")


print("═" * 70)
print("  Sequential Forward Selection -- GRU on Hybrid-Normalized data")
print(f"  Params      : {best_params}")
print(f"  Runs/combo  : {N_RUNS}")
print(f"  Total combos: {sum(range(1, len(feature_names) + 1))}  "
      f"(vs. {2 ** len(feature_names) - 1} for full exhaustive search)")
print(f"  Results file: {os.path.abspath(RESULTS_FILE)}")
print("═" * 70)

selected  = []
remaining = feature_names.copy()

all_combo_results = []   # every combination evaluated (55 total)
sfs_path           = []  # the greedy path actually taken (10 entries)

for round_idx in range(1, len(feature_names) + 1):

    print(f"\n{'─' * 70}")
    print(f"Round {round_idx}/{len(feature_names)}  |  selected so far: {selected or '(none)'}")
    print(f"{'─' * 70}")

    round_candidates = []

    for cand in remaining:
        trial_features = selected + [cand]
        idxs = [feature_index[f] for f in trial_features]

        X_tr_sub = X_train[:, :, idxs]
        X_va_sub = X_val[:,   :, idxs]
        X_te_sub = X_test[:,  :, idxs]

        run_metrics = []
        for run in range(1, N_RUNS + 1):
            metrics = train_and_evaluate_gru(
                copy.deepcopy(best_params),
                X_tr_sub, y_train,
                X_va_sub, y_val,
                X_te_sub, y_test,
                class_ratio=class_ratio
            )[0]
            append_result(round_idx, trial_features, run, metrics)
            run_metrics.append(metrics)

        avg_tss = float(np.mean([m["tss"] for m in run_metrics]))
        avg_brier = float(np.mean([m["brier"] for m in run_metrics]))

        round_candidates.append({
            "round": round_idx,
            "added": cand,
            "features": trial_features,
            "avg_tss": avg_tss,
            "avg_brier": avg_brier,
        })
        all_combo_results.append(round_candidates[-1])

        print(f"  +{cand:<3} -> {trial_features}  |  avg TSS={avg_tss:.4f}  avg Brier={avg_brier:.4f}")

    winner = max(round_candidates, key=lambda r: r["avg_tss"])
    selected = winner["features"]
    remaining.remove(winner["added"])
    sfs_path.append(winner)

    print(f"\n  >>> Round {round_idx} winner: +{winner['added']}  "
          f"(avg TSS={winner['avg_tss']:.4f})  ->  {selected}")

print(f"\n{'═' * 70}")
print("SFS complete.")
print(f"All {len(all_combo_results)} combinations saved -> {os.path.abspath(RESULTS_FILE)}")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
  Sequential Forward Selection -- GRU on Hybrid-Normalized data
  Params      : {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
  Runs/combo  : 5
  Total combos: 55  (vs. 1023 for full exhaustive search)
  Results file: ./results/gru_feature_selection_sfs.txt
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Round 1/10  |  selected so far: (none)
──────────────────────────────────────────────────────────────────────
  +F   -> ['F']  |  avg TSS=0.0387  avg Brier=0.2091
  +Np  -> ['Np']  |  avg TSS=-0.0098  avg Brier=0.2807
  +P4  -> ['P4']  |  avg TSS=0.5893  avg Brier=0.1312
  +P5  -> ['P5']  |  avg TSS=0.3083  avg Brier=0.2074
  +P6  -> ['P6']  |  avg TSS=0.1663  avg Brier=0.2226
  +Tp  -> ['Tp']  |  avg TSS=0.0093  avg Brier=0.2357
  +V   -> ['V']  |  avg TSS=0.0097  avg Brier=0.2398
  +Vx  -> ['V

## 4. Results Summary

In [4]:
# ---------------------------------------------------------
# Sort every combination evaluated (all 55, not just the greedy path) by
# average TSS, and report the overall best -- which may be smaller than
# the full 10-feature set the greedy path ends at.
# ---------------------------------------------------------
ranked = sorted(all_combo_results, key=lambda r: r["avg_tss"], reverse=True)

print("Top 15 feature combinations by average TSS (out of "
      f"{len(all_combo_results)} evaluated):")
print(f"{'Rank':<5} {'Round':<6} {'Avg TSS':>8} {'Avg Brier':>10}  Features")
print("─" * 100)
for rank, r in enumerate(ranked[:15], start=1):
    print(f"{rank:<5} {r['round']:<6} {r['avg_tss']:>8.4f} {r['avg_brier']:>10.4f}  {r['features']}")

best = ranked[0]
print("\nBest feature combination overall:")
print(f"  Features : {best['features']}")
print(f"  Count    : {len(best['features'])}")
print(f"  Avg TSS  : {best['avg_tss']:.4f}")
print(f"  Avg Brier: {best['avg_brier']:.4f}")

print("\nGreedy SFS path (one winner per round):")
print(f"{'Round':<6} {'Added':<6} {'Avg TSS':>8}  Cumulative features")
print("─" * 90)
for r in sfs_path:
    print(f"{r['round']:<6} {r['added']:<6} {r['avg_tss']:>8.4f}  {r['features']}")

print(f"\nFull per-run results: {os.path.abspath(RESULTS_FILE)}")

Top 15 feature combinations by average TSS (out of 55 evaluated):
Rank  Round   Avg TSS  Avg Brier  Features
────────────────────────────────────────────────────────────────────────────────────────────────────
1     3        0.6753     0.1035  ['P4', 'Xl', 'Vx']
2     4        0.6660     0.1100  ['P4', 'Xl', 'Vx', 'Np']
3     4        0.6649     0.1123  ['P4', 'Xl', 'Vx', 'F']
4     3        0.6647     0.0999  ['P4', 'Xl', 'P5']
5     7        0.6640     0.1034  ['P4', 'Xl', 'Vx', 'Np', 'F', 'Tp', 'P6']
6     5        0.6612     0.1126  ['P4', 'Xl', 'Vx', 'Np', 'F']
7     5        0.6605     0.0962  ['P4', 'Xl', 'Vx', 'Np', 'V']
8     3        0.6598     0.1197  ['P4', 'Xl', 'V']
9     4        0.6597     0.0921  ['P4', 'Xl', 'Vx', 'P5']
10    6        0.6569     0.1309  ['P4', 'Xl', 'Vx', 'Np', 'F', 'Tp']
11    7        0.6540     0.1079  ['P4', 'Xl', 'Vx', 'Np', 'F', 'Tp', 'V']
12    10       0.6530     0.1118  ['P4', 'Xl', 'Vx', 'Np', 'F', 'Tp', 'P6', 'P5', 'V', 'Xs']
13    6       

## 5. Save Best-Feature Dataset

In [5]:
# ---------------------------------------------------------
# Save a new dataset containing only the best feature combination found
# by SFS (`ranked[0]`), in the same {'X', 'y'} pickle format used
# throughout the rest of the pipeline. This is what notebook 3 (and
# everything downstream) should load instead of the full 10-feature set.
# ---------------------------------------------------------
best_features = best["features"]
best_idxs     = [feature_index[f] for f in best_features]

X_train_best = X_train[:, :, best_idxs]
X_val_best   = X_val[:,   :, best_idxs]
X_test_best  = X_test[:,  :, best_idxs]

best_features_dir = "./final_split_data_HybridNorm_BestFeatures"
os.makedirs(best_features_dir, exist_ok=True)

def save_bundle(X, y, path):
    with open(path, "wb") as f:
        pickle.dump({"X": X, "y": y}, f)
    print(f"Saved: {path}")

save_bundle(X_train_best, y_train, os.path.join(best_features_dir, "train_set.pkl"))
save_bundle(X_val_best,   y_val,   os.path.join(best_features_dir, "val_set.pkl"))
save_bundle(X_test_best,  y_test,  os.path.join(best_features_dir, "test_set.pkl"))

with open(os.path.join(best_features_dir, "selected_features.txt"), "w") as f:
    f.write(",".join(best_features) + "\n")
    f.write(f"avg_tss={best['avg_tss']:.6f}\n")
    f.write(f"avg_brier={best['avg_brier']:.6f}\n")

print(f"\nSaved best-feature dataset ({len(best_features)} features: {best_features})")
print(f"  -> {os.path.abspath(best_features_dir)}/")
print(f"Train: {X_train_best.shape} | Val: {X_val_best.shape} | Test: {X_test_best.shape}")

Saved: ./final_split_data_HybridNorm_BestFeatures/train_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures/val_set.pkl
Saved: ./final_split_data_HybridNorm_BestFeatures/test_set.pkl

Saved best-feature dataset (3 features: ['P4', 'Xl', 'Vx'])
  -> ./final_split_data_HybridNorm_BestFeatures/
Train: (12455, 288, 3) | Val: (1780, 288, 3) | Test: (3559, 288, 3)
